# CS604 — Qian’s prefix-tuning ablation at 336

Use an **A100 40GB GPU**. The experiment freezes the original backbone and trains independent K/V prefixes plus a **prototype-initialized linear classifier**. Prefix parameters must remain **strictly below 5% of the original backbone**, excluding the classifier.

The main screen is **9 arms (length 5/20/50 × last block/last 4/all 12 blocks) plus classifier-only**, with 5 epochs each and no time cutoff. This can require multiple Colab sessions. Drive stores results and completed-arm markers; rerunning skips completed matching arms and restarts interrupted arms.

The notebook prepares a fixed 40-fit/5-holdout split (seed 42), builds fit-only 336 prototypes, runs a 3-epoch/1,000-class LR pilot, then screens the arms using its selected LR. This deterministic split is recorded explicitly; it is not asserted to reproduce an earlier notebook's unpublished split manifests.

Set up `CS604_RUNS` in My Drive as before. The branch must be available on GitHub before cloning it in Colab. Never commit your GitHub token.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_prefix_tuning_ablation"
TEAM = "08"
CONFIG = "configs/prefix_ablation_336.yaml"
# Reuse this name across sessions. Change it when changing the recipe.
EXPERIMENT_NAME = "qq_prefix_ablation_336_seed42_v1"
EXPERIMENT_DIR = f"{DRIVE_OUT}/{EXPERIMENT_NAME}"
SPLIT_DIR = f"/content/{EXPERIMENT_NAME}_splits"
RUN_PILOT = True
RUN_SCREEN = True
PREFIX_LR = None  # None uses the persisted pilot choice; explicit override starts a new recipe.
ARM_FILTER = None  # None runs all 10; e.g. ["classifier_only", "all_p20"] for one session.
RUN_CONFIRMATION = False
CONFIRM_ARMS = ["classifier_only"]  # Add the best two prefix arms after inspecting results.
CONFIRM_SEEDS = [42, 43, 44]
RUN_LATENCY = True
SELECTED_ARM = None  # Set after confirmation, e.g. "all_p20", to export a final recipe.
FINAL_RUN_NAME = None  # Optional: folder name under EXPERIMENT_DIR/final after final training.


## 1. Clone or pull the repo

In [ ]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
subprocess.run(["python", "scripts/download_data.py", "--dest", DATA_DIR], check=True)


## 4b. Prepare the fixed split and review the experiment

Manifests are saved on Drive; local split roots symlink images from `/content/data`. Full-scale arms use 400,000 fitting and 50,000 holdout images. The pilot selects 1,000 classes from this same split, retaining the 10,000-way classifier. Absent pilot classes have zero prototype rows.

Prototypes are built separately for pilot and full fitting sets at 336. All arms within a stage reuse identical prototypes. The old all-data `prototypes.pt` is never used for holdout selection.


In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
import torch
from src.utils import load_config, apply_overrides
from src.peft.prefix_ablation import prepare, arm_grid, run_stage, export_final_config

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
BASE = load_config(CONFIG)
apply_overrides(BASE, data_root=DATA_DIR, output_dir=EXPERIMENT_DIR)
assert BASE.img_size == 336 and BASE.head_init == "prototype" and BASE.head_type == "linear"
assert BASE.max_train_minutes is None and BASE.epochs == 5
EXPERIMENT = Path(EXPERIMENT_DIR)
ROOTS = prepare(BASE, EXPERIMENT, split_dir=SPLIT_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Experiment:", EXPERIMENT)
print("Split identity:", json.loads((EXPERIMENT / "experiment.json").read_text()))
display(pd.DataFrame(list(BASE.to_dict().items()), columns=["parameter", "value"]))
rows = []
for arm in arm_grid(BASE):
    blocks = {"none": 0, "last1": 1, "last4": 4, "all": 12}[arm["prefix_placement"]]
    count = 2 * blocks * arm["prefix_length"] * 768
    rows.append({**arm, "prefix_parameters": count, "backbone_trainable_pct": 100 * count / 85986816})
display(pd.DataFrame(rows))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(EXPERIMENT / "launcher_commit.txt").write_text(commit + "\n")


## 5. Learning-rate pilot

Three 3-epoch runs: prefix LR `1e-4`, `3e-4`, `1e-3`, all 12 blocks, length 20. Classifier LR stays `1e-3`. Choose by final holdout accuracy; within 0.3 percentage points, prefer the lower stable LR. The selected LR is persisted in `pilot_selection.json`.

All runs use AdamW, zero weight decay, BF16, 5% warmup then cosine decay, effective batch 64. Prototype extraction and evaluation are outside training-only timing. Stop and adjust a new experiment recipe if memory does not fit; do not change batch size halfway through a grid.


In [ ]:
if RUN_PILOT:
    PILOT_RESULTS = run_stage(BASE, EXPERIMENT, ROOTS, "pilot")
    display(pd.DataFrame(PILOT_RESULTS))
if (EXPERIMENT / "pilot_selection.json").exists():
    print("Pilot selection:", json.loads((EXPERIMENT / "pilot_selection.json").read_text()))


## 6. Main length × placement screen

Each arm starts from pretrained weights and the same fit-only prototype head. Only prefixes and classifier receive gradients; actual trainable counts are checked before training. Each arm gets 5 complete epochs with the same data seed, not a shared wall-clock cutoff. Initial, best, and final accuracy are retained.

Use `ARM_FILTER` to spread the grid across sessions. Completed arms are reused only for an identical recipe and data. Changing LR, seed, batch size or schedule requires a new experiment name. Interrupted arms restart; optimizer-level resume is not implemented.


In [ ]:
SCREEN_RESULTS = []
if RUN_SCREEN:
    SCREEN_RESULTS = run_stage(BASE, EXPERIMENT, ROOTS, "screen", arms=ARM_FILTER, prefix_lr=PREFIX_LR)
# Read every completed arm, including arms run in earlier sessions.
completed = sorted((EXPERIMENT / "runs" / "screen").glob("*/seed*/completed.json"))
SCREEN_RESULTS = [json.loads(p.read_text()) for p in completed]
if SCREEN_RESULTS:
    SCREEN_TABLE = pd.DataFrame(SCREEN_RESULTS).sort_values("best_val_acc", ascending=False)
    SCREEN_TABLE.to_csv(EXPERIMENT / "screen_results.csv", index=False)
    display(SCREEN_TABLE)
    print("Best single-seed result is provisional; inspect curves and confirm across seeds.")


## 7. Optional confirmation across seeds

After the screen, set `CONFIRM_ARMS` to classifier-only plus the best two prefix arms, then enable `RUN_CONFIRMATION`. Use three seeds total. The split and prototype head stay fixed; training/prefix initialization seeds vary. Inspect mean accuracy and variation before selecting an architecture. If finalists are still improving, extend all finalists and the baseline under a new shared recipe.


In [ ]:
CONFIRM_RESULTS = []
if RUN_CONFIRMATION:
    assert len(set(CONFIRM_ARMS) - {"classifier_only"}) == 2, "Select the best two prefix arms plus baseline."
    assert "classifier_only" in CONFIRM_ARMS
    CONFIRM_RESULTS = run_stage(BASE, EXPERIMENT, ROOTS, "confirm", arms=CONFIRM_ARMS,
                               seeds=CONFIRM_SEEDS, prefix_lr=PREFIX_LR)
    table = pd.DataFrame(CONFIRM_RESULTS)
    display(table)
    display(table.groupby("arm")["best_val_acc"].agg(["mean", "std", "count"]))


## 8. Loss curves

Plot the completed screen arms. Validation curves use only the internal holdout. Compare initial/best/final accuracy and loss; five epochs do not establish convergence.


In [ ]:
from IPython.display import display, Image

for result in SCREEN_RESULTS:
    run_dir = Path(result["run_dir"])
    figures = run_dir / "figures"
    subprocess.run(["python", "scripts/plot_curves.py", "--run_dir", str(run_dir),
                    "--out_dir", str(figures)], check=True)
    print(result["arm"])
    for plot in sorted(figures.glob("*.png")):
        display(Image(filename=str(plot)))


## 9. Inference latency

Measure each completed screen arm’s final checkpoint on the same GPU, using synthetic inputs at batch sizes 1 and 64, BF16, 30 warm-ups and 200 iterations. Save `inference_latency.json` in each run. Results exclude preprocessing and transfers. Disable `RUN_LATENCY` while running a partial grid if needed.


In [ ]:
from src.model import build_model
from src.predict import load_checkpoint
from src.utils import set_seed
from scripts.benchmark_latency import measure

if RUN_LATENCY:
    for result in SCREEN_RESULTS:
        run_dir = Path(result["run_dir"])
        cfg = load_config(run_dir / "config.yaml")
        cfg.pretrained = True  # Required for partial prefix checkpoints.
        set_seed(int(cfg.seed))
        device = torch.device("cuda")
        model = build_model(cfg).to(device).eval()
        load_checkpoint(model, run_dir / "last.pt", device)
        latency = {"checkpoint": str(run_dir / "last.pt"), "gpu": torch.cuda.get_device_name(0),
                   "torch_version": str(torch.__version__), "img_size": int(cfg.img_size),
                   "precision": "bf16" if cfg.bf16 else "fp32", "warmup_iterations": 30,
                   "timed_iterations": 200, "scope": "Synthetic forward pass only", "measurements": []}
        try:
            for bs in (1, 64):
                batch = torch.randn(bs, 3, cfg.img_size, cfg.img_size, device=device)
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=bool(cfg.bf16)):
                    ms, _ = measure(model, batch, device, 30, 200)
                latency["measurements"].append({"batch_size": bs, "ms_per_image": ms, "ms_per_batch": ms * bs})
                del batch
        finally:
            del model
            torch.cuda.empty_cache()
        (run_dir / "inference_latency.json").write_text(json.dumps(latency, indent=2))
        print(result["arm"], latency["measurements"])


## 10. Export the selected all-data recipe (no automatic final training)

After confirming a winner, set `SELECTED_ARM`. Export a fresh 144-training-minute configuration using all 450,000 labelled images, with 5% time-based warmup and cosine decay. All-data prototypes must be rebuilt separately at 336. This cell only exports the configuration; it does not train or submit.

To run the final experiment afterward, use the printed commands. The final run has no internal holdout; official validation is evaluated through predictions/CodaBench. Retain the declared seed and budget before seeing its score.


In [ ]:
if SELECTED_ARM is not None:
    final_config = export_final_config(BASE, EXPERIMENT, SELECTED_ARM, prefix_lr=PREFIX_LR)
    final = load_config(final_config)
    print("Final recipe:", final_config)
    print("Build all-data prototypes in a separate cell before final training:")
    print("from src.peft.prefix_ablation import ensure_prototypes")
    print(f'ensure_prototypes(load_config("{final_config}"))')
    print("Train final run:")
    print(f'python -u -m src.train --config "{final_config}"')
else:
    print("Select an arm after reviewing screen/confirmation results.")


## 11. Optional official predictions after the final run

Set `FINAL_RUN_NAME` to a completed folder under this experiment’s `final/` directory. Predictions use that run’s saved configuration and `last.pt`, overriding the data root to the original official dataset. Development checkpoints are not submitted automatically.


In [ ]:
if FINAL_RUN_NAME is not None:
    import shutil
    RUN_DIR = EXPERIMENT / "final" / FINAL_RUN_NAME
    ckpt = RUN_DIR / "last.pt"
    assert ckpt.is_file(), f"Checkpoint missing: {ckpt}"
    final_cfg = load_config(RUN_DIR / "config.yaml")
    assert final_cfg.data_root == DATA_DIR, "Only use the all-data final run here."
    for split in ("val", "test"):
        if not (Path(DATA_DIR) / f"{split}.txt").exists():
            continue
        subprocess.run(["python", "-m", "src.predict", "--config", str(RUN_DIR / "config.yaml"),
                        "--ckpt", str(ckpt), "--split", split, "--team", TEAM,
                        "--data_root", DATA_DIR, "--out_dir", str(RUN_DIR)], check=True)
        shutil.make_archive(str(RUN_DIR / f"{split}_team_{TEAM}"), "zip",
                            root_dir=str(RUN_DIR), base_dir=f"{split}_team_{TEAM}.txt")
else:
    print("No final checkpoint selected; skipping official predictions.")
